# Tarea 1: Extracción de datos fenotípicos — Metaanálisis de metilación en esquizofrenia

Datasets incluidos:
- GSE116378 — Boks et al., 2018 (DUTCHSCZ, 15 SCZ / 49 CTR)
- GSE116379 — Boks et al., 2018 (DUTCHSCZ con hambruna, ~153 muestras)
- GSE80417  — Hannon et al., 2016 (UCL, 353 SCZ / 322 CTR)
- GSE84727  — Hannon et al., 2016 (Aberdeen, 414 SCZ / 433 CTR)
- GSE152027 — Hannon et al., 2021 (IoPPN, 290 SCZ / 203 CTR + 307 FEP)
- GSE147221 — Hannon et al., 2021 (Dublin, 364 SCZ / 349 CTR)

Plataforma: Illumina HumanMethylation450 BeadChip (450K)
Fuente: NCBI Gene Expression Omnibus (GEO)
Objetivo: Extraer los datos fenotípicos de cada dataset, equivalente al phenoData de R con GEOquery.

In [2]:
import GEOparse    # Libreria para descargar y parsear datos de NCBI GEO (equivalente a GEOquery en R)
import pandas as pd    # Libreria para manejo de datos tabulares (equivalente a data.frame en R)
import os    # Libreria del sistema para crear carpetas y manejar rutas

# Lista de todos los datasets del metaanalisis con su descripcion
# Cada entrada es una tupla: (codigo_GSE, referencia_breve)
DATASETS = [
    ("GSE116378", "Boks_2018_DUTCHSCZ_sangre"),
    ("GSE116379", "Boks_2018_DUTCHSCZ_hambruna"),
    ("GSE80417",  "Hannon_2016_UCL_sangre"),
    ("GSE84727",  "Hannon_2016_Aberdeen_sangre"),
    ("GSE152027", "Hannon_2021_IoPPN_sangre"),
    ("GSE147221", "Hannon_2021_Dublin_sangre"),
]

# Crear la carpeta de resultados en la raiz del proyecto si no existe
# exist_ok=True evita error si la carpeta ya fue creada previamente
os.makedirs("../resultados", exist_ok=True)

# Funcion para desempaquetar la columna characteristics_ch1
# En GEO, esta columna almacena las variables clinicas como una lista de strings
# con formato "clave: valor", por ejemplo: ["tissue: Whole blood", "group: CTR", "age: 27"]
# Equivalente al parsing manual de phenoData en R con GEOquery
def parsear_characteristics(lista):

    # Diccionario donde se almacenaran los pares clave-valor extraidos
    resultado = {}

    # Verifica que el valor sea efectivamente una lista antes de procesarla
    if isinstance(lista, list):

        # Recorre cada elemento de la lista, por ejemplo "group: CTR"
        for item in lista:

            # Solo procesa los elementos que contienen el separador ":"
            if ":" in item:

                # Divide el string en clave y valor por el primer ":" encontrado
                # El argumento 1 limita la division a un solo corte, por si el valor
                # contiene ":" internamente
                clave, valor = item.split(":", 1)

                # Guarda el par en el diccionario eliminando espacios sobrantes
                resultado[clave.strip()] = valor.strip()

    return resultado

# Funcion principal que procesa un dataset completo dado su codigo GSE
# Retorna el DataFrame de phenoData limpio y lo guarda como CSV individual
def procesar_dataset(gse_id, nombre_referencia):

    print(f"\n{'='*60}")
    print(f"Procesando: {gse_id} — {nombre_referencia}")
    print(f"{'='*60}")

    # Descarga y parsea el archivo SOFT desde NCBI GEO
    # Si el archivo ya fue descargado previamente, lo carga desde disco sin volver a descargarlo
    # Equivalente a: gse <- getGEO("GSExxxxxx") en R
    gse = GEOparse.get_GEO(geo=gse_id, destdir="./data_geo")

    # Confirma el numero total de muestras descargadas
    print(f"Numero de muestras: {len(gse.gsms)}")

    # Inicializa la lista donde se acumulara un diccionario por cada muestra
    registros = []

    # Itera sobre cada muestra del dataset
    # gse.gsms es un diccionario: clave = identificador GSM, valor = objeto con metadatos
    for nombre_gsm, muestra in gse.gsms.items():

        # Crea un diccionario para esta muestra con su identificador GSM
        fila = {"gsm_id": nombre_gsm}

        # Recorre todos los campos de metadatos disponibles para esta muestra
        # muestra.metadata es un diccionario donde cada valor es una lista de strings
        for campo, valor in muestra.metadata.items():

            # Si la lista tiene un solo elemento, lo extrae directamente
            # Si tiene varios, conserva la lista completa
            fila[campo] = valor[0] if isinstance(valor, list) and len(valor) == 1 else valor

        registros.append(fila)

    # Convierte la lista de diccionarios en un DataFrame de pandas
    # Equivalente a: do.call(rbind, lapply(...)) en R
    pheno_data = pd.DataFrame(registros)

    # Selecciona las columnas de identificacion y las clinicas
    # Todas las variables clinicas del estudio van siempre en campos llamados "characteristics_chX"
    # por estandar del formato SOFT de GEO, garantizando que ningun dato fenotipico quede fuera
    columnas_clinicas = [
        col for col in pheno_data.columns
        if "characteristics" in col or col in ("gsm_id", "title", "platform_id")
    ]

    pheno_limpio = pheno_data[columnas_clinicas].copy()

    # Desempaqueta characteristics_ch1 en columnas individuales (tissue, group, gender, age, etc.)
    if "characteristics_ch1" in pheno_limpio.columns:
        caracteristicas = pheno_limpio["characteristics_ch1"].apply(parsear_characteristics)
        caracteristicas_df = pd.DataFrame(caracteristicas.tolist())

        # Construye el DataFrame final: identificadores + variables clinicas desempaquetadas
        cols_id = [c for c in ["gsm_id", "title", "platform_id"] if c in pheno_limpio.columns]
        pheno_final = pd.concat(
            [pheno_limpio[cols_id].reset_index(drop=True),
             caracteristicas_df.reset_index(drop=True)],
            axis=1    # axis=1 indica concatenacion por columnas, no por filas
        )
    else:
        # Si no existe characteristics_ch1, usa el dataframe tal como esta
        pheno_final = pheno_limpio.copy()

    # Agrega una columna con el codigo GSE para identificar el dataset de origen
    # Esto es especialmente util en el CSV combinado del metaanalisis
    pheno_final.insert(0, "gse_id", gse_id)

    # Agrega una columna con el nombre de referencia del estudio
    pheno_final.insert(1, "referencia", nombre_referencia)

    # Muestra un resumen del resultado
    print(f"Dimensiones del phenoData: {pheno_final.shape}")
    print(f"Columnas: {pheno_final.columns.tolist()}")
    print(pheno_final.head(3).to_string())

    # Guarda el CSV individual de este dataset
    ruta = f"../resultados/{gse_id}_phenodata.csv"
    pheno_final.to_csv(ruta, index=False)
    print(f"Guardado en: {ruta}")

    return pheno_final

# Procesa todos los datasets e inicializa la lista para el CSV combinado
todos_los_datos = []

for gse_id, nombre_referencia in DATASETS:
    try:
        df = procesar_dataset(gse_id, nombre_referencia)
        todos_los_datos.append(df)
    except Exception as e:
        # Si un dataset falla (por ejemplo por problemas de red), registra el error
        # y continua con los demas sin detener el proceso completo
        print(f"ERROR procesando {gse_id}: {e}")

# Combina todos los DataFrames en uno solo para el metaanalisis
# Las columnas que no existan en algun dataset quedan como NaN (equivalente a NA en R)
pheno_combinado = pd.concat(todos_los_datos, axis=0, ignore_index=True)

print(f"\n{'='*60}")
print("RESUMEN DEL METAANALISIS")
print(f"{'='*60}")
print(f"Total de muestras combinadas: {len(pheno_combinado)}")
print(f"Distribucion por dataset:")
print(pheno_combinado.groupby("gse_id").size().to_string())

# Guarda el CSV combinado con todos los datasets
ruta_combinado = "../resultados/metaanalisis_phenodata_combinado.csv"
pheno_combinado.to_csv(ruta_combinado, index=False)
print(f"\nCSV combinado guardado en: {ruta_combinado}")

17-Apr-2026 22:52:08 DEBUG utils - Directory ./data_geo already exists. Skipping.
17-Apr-2026 22:52:08 INFO GEOparse - File already exist: using local version.
17-Apr-2026 22:52:08 INFO GEOparse - Parsing ./data_geo/GSE116378_family.soft.gz: 
17-Apr-2026 22:52:08 DEBUG GEOparse - DATABASE: GeoMiame
17-Apr-2026 22:52:08 DEBUG GEOparse - SERIES: GSE116378
17-Apr-2026 22:52:08 DEBUG GEOparse - PLATFORM: GPL13534



Procesando: GSE116378 — Boks_2018_DUTCHSCZ_sangre


/home/xkqax/proyectos/epigenetica_computacional/venv_tarea1/lib/python3.12/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (0: CHR, 1: Chromosome_36, 2: Coordinate_36, 3: SPOT_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")
17-Apr-2026 22:52:28 DEBUG GEOparse - SAMPLE: GSM3230147
17-Apr-2026 22:52:28 DEBUG GEOparse - SAMPLE: GSM3230148
17-Apr-2026 22:52:28 DEBUG GEOparse - SAMPLE: GSM3230149
17-Apr-2026 22:52:28 DEBUG GEOparse - SAMPLE: GSM3230150
17-Apr-2026 22:52:28 DEBUG GEOparse - SAMPLE: GSM3230151
17-Apr-2026 22:52:28 DEBUG GEOparse - SAMPLE: GSM3230152
17-Apr-2026 22:52:28 DEBUG GEOparse - SAMPLE: GSM3230153
17-Apr-2026 22:52:28 DEBUG GEOparse - SAMPLE: GSM3230154
17-Apr-2026 22:52:28 DEBUG GEOparse - SAMPLE: GSM3230155
17-Apr-2026 22:52:28 DEBUG GEOparse - SAMPLE: GSM3230156
17-Apr-2026 22:52:28 DEBUG GEOparse - SAMPLE: GSM3230157
17-Apr-2026 22:52:28 DEBUG GEOparse - SAMPLE:

Numero de muestras: 64
Dimensiones del phenoData: (64, 16)
Columnas: ['gse_id', 'referencia', 'gsm_id', 'title', 'platform_id', 'tissue', 'group', 'gender', 'age', 'cd8t', 'cd4t', 'nk', 'bcell', 'mono', 'sentrix', 'position']
      gse_id                 referencia      gsm_id                         title platform_id       tissue group gender age                cd8t               cd4t                  nk               bcell                mono     sentrix position
0  GSE116378  Boks_2018_DUTCHSCZ_sangre  GSM3230147  DUTCHSCZ-01: Blood_Control_1    GPL13534  Whole blood   CTR      F  27   0.114405658860362  0.182269209534099  0.0405247085359864  0.0821755884025784   0.104378038771865  9373550151   R02C02
1  GSE116378  Boks_2018_DUTCHSCZ_sangre  GSM3230148  DUTCHSCZ-02: Blood_Control_2    GPL13534  Whole blood   CTR      F  24  0.0416757400422807   0.12448009016605  0.0517446507702588  0.0349972112175631  0.0749324122133966  9373550151   R03C01
2  GSE116378  Boks_2018_DUTCHSCZ_sangre  G

17-Apr-2026 22:52:28 DEBUG utils - Directory ./data_geo already exists. Skipping.
17-Apr-2026 22:52:28 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE116nnn/GSE116379/soft/GSE116379_family.soft.gz to ./data_geo/GSE116379_family.soft.gz


...

Numero de muestras: 800
Dimensiones del phenoData: (800, 8)
Columnas: ['gse_id', 'referencia', 'gsm_id', 'title', 'platform_id', 'status', 'gender', 'ageatbloodcollection']
      gse_id                referencia      gsm_id                       title platform_id status gender ageatbloodcollection
0  GSE152027  Hannon_2021_IoPPN_sangre  GSM4599914  10003885059_R01C01 Sample1    GPL13534    CON      F                   NA
1  GSE152027  Hannon_2021_IoPPN_sangre  GSM4599915  10003885059_R01C02 Sample2    GPL13534    FEP      M                   22
2  GSE152027  Hannon_2021_IoPPN_sangre  GSM4599916  10003885059_R02C01 Sample3    GPL13534    CON      F                   23
Guardado en: ../resultados/GSE152027_phenodata.csv


17-Apr-2026 22:54:00 DEBUG utils - Directory ./data_geo already exists. Skipping.
17-Apr-2026 22:54:00 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE147nnn/GSE147221/soft/GSE147221_family.soft.gz to ./data_geo/GSE147221_family.soft.gz



Procesando: GSE147221 — Hannon_2021_Dublin_sangre


100%|██████████| 64.7M/64.7M [00:05<00:00, 13.2MB/s]  
17-Apr-2026 22:54:06 DEBUG downloader - Size validation passed
17-Apr-2026 22:54:06 DEBUG downloader - Moving /tmp/tmpr9rn563l to /home/xkqax/proyectos/epigenetica_computacional/notebooks/data_geo/GSE147221_family.soft.gz
17-Apr-2026 22:54:06 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE147nnn/GSE147221/soft/GSE147221_family.soft.gz
17-Apr-2026 22:54:06 INFO GEOparse - Parsing ./data_geo/GSE147221_family.soft.gz: 
17-Apr-2026 22:54:06 DEBUG GEOparse - DATABASE: GeoMiame
17-Apr-2026 22:54:06 DEBUG GEOparse - SERIES: GSE147221
17-Apr-2026 22:54:06 DEBUG GEOparse - PLATFORM: GPL13534
/home/xkqax/proyectos/epigenetica_computacional/venv_tarea1/lib/python3.12/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (0: CHR, 1: Chromosome_36, 2: Coordinate_36, 3: SPOT_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, 

Numero de muestras: 720
Dimensiones del phenoData: (720, 9)
Columnas: ['gse_id', 'referencia', 'gsm_id', 'title', 'platform_id', 'Sex', 'consensus_diagnosis', 'status', 'age']
      gse_id                 referencia      gsm_id                                              title platform_id Sex consensus_diagnosis   status   age
0  GSE147221  Hannon_2021_Dublin_sangre  GSM4420456  200379120027_R01C01: genomic DNA from whole Blood    GPL13534   M       Schizophrenia     Case  37.6
1  GSE147221  Hannon_2021_Dublin_sangre  GSM4420457  200379120027_R02C01: genomic DNA from whole Blood    GPL13534   M             Control  Control    23
2  GSE147221  Hannon_2021_Dublin_sangre  GSM4420458  200379120027_R03C01: genomic DNA from whole Blood    GPL13534   M             Control  Control    45
Guardado en: ../resultados/GSE147221_phenodata.csv

RESUMEN DEL METAANALISIS
Total de muestras combinadas: 3259
Distribucion por dataset:
gse_id
GSE116378     64
GSE116379    153
GSE147221    720
GSE152027   